# IC-272 Lab 5

Standardisation and Ridge Regression


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
df = pd.read_csv("ic272_lab5_rent.csv")

features = [c for c in df.columns if c != "rent"]
X = df[features].to_numpy(float)
y = df["rent"].to_numpy(float)

idx = np.random.default_rng(0).permutation(len(y))
tr, te = idx[:40], idx[40:]

X_train, X_test = X[tr], X[te]
y_train, y_test = y[tr], y[te]

print("train:", X_train.shape, "test:", X_test.shape)
print(features)


train: (40, 13) test: (260, 13)
['area', 'rooms', 'age', 'dist_km', 'floors', 'noise_a', 'noise_b', 'noise_c', 'noise_d', 'noise_id', 'noise_tiny1', 'noise_tiny2', 'noise_tiny3']


In [3]:
def design_matrix(X):
    return np.c_[np.ones(len(X)), X]


def fit_normal_equation(X, y):
    A = design_matrix(X)
    return np.linalg.solve(A.T @ A, A.T @ y)


def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


## Task 1

In [ ]:
theta_raw = fit_normal_equation(X_train, y_train)

print(f'{"feature":14s}{"coefficient":13s}{"column sd":12s}')

for name, coef, s in zip( features , theta_raw[1:] , X_train.std(axis=0) ):
    print(f"{name:14s}{coef:13.4f}{s:12.4f}")


feature       coefficient  column sd   
area                 6.2035      1.3899
rooms                3.5651      1.4239
age                 -1.4889      2.2148
dist_km             -2.1478      2.2604
floors               1.9149      0.8182
noise_a             -0.4443      2.0689
noise_b             -0.1819      2.3947
noise_c             -0.3990      2.0424
noise_d              0.1087      1.9085
noise_id            -0.0004   1620.1592
noise_tiny1        -11.2210      0.0563
noise_tiny2         -6.3696      0.0462
noise_tiny3        -11.9928      0.0526


In [5]:
def standardise(X_train, X_test):
    mu = np.mean(X_train, axis=0)
    sigma = np.std(X_train, axis=0)

    X_train_std = (X_train - mu) / sigma
    X_test_std = (X_test - mu) / sigma

    return X_train_std, X_test_std, mu, sigma


In [6]:
X_train_std, X_test_std, mu, sigma = standardise(
    X_train,
    X_test
)

print("Training mean:")
print(np.mean(X_train_std, axis=0))

print("Training standard deviation:")
print(np.std(X_train_std, axis=0))

print("Test mean:")
print(np.mean(X_test_std, axis=0))

print("Test standard deviation:")
print(np.std(X_test_std, axis=0))


Training mean:
[-2.88657986e-16  5.82867088e-17 -4.57966998e-16  3.05311332e-16
 -5.55111512e-17 -2.77555756e-18 -1.66533454e-17 -1.94289029e-17
 -3.46944695e-17  1.57207580e-14  4.44089210e-17  1.66533454e-17
 -1.59594560e-17]
Training standard deviation:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Test mean:
[ 0.04138349  0.32143501 -0.25106447  0.23094469  0.08696883  0.12342875
  0.26003852  0.01859972 -0.16942614  0.00298024 -0.12479629 -0.23997969
  0.28657726]
Test standard deviation:
[0.90471265 0.99222493 0.98564    1.02481282 0.99700378 0.97260292
 0.78643272 1.0231371  1.06495137 0.89446554 0.885202   1.03412701
 0.8849837 ]


## Task 2

In [8]:
def fit_ridge(X, y, lam):
    A = design_matrix(X)
    p = A.shape[1]

    I = np.eye(p)
    I[0, 0] = 0

    M = A.T @ A + lam * I
    b = A.T @ y

    theta = np.linalg.solve(M, b)

    return theta


In [9]:
lams = [0, 1, 10, 100]

for lam in lams:
    theta = fit_ridge(
        X_train_std,
        y_train,
        lam
    )

    print(f"lambda: {lam}")
    print("coefficients:")
    print(theta)

    zero_cnt = np.sum(np.abs(theta[1:]) < 1e-8)
    print("number of zero coefficients:", zero_cnt)
    print()


lambda: 0
coefficients:
[52.7675      8.62228061  5.07634651 -3.29768683 -4.85508405  1.56669972
 -0.91915016 -0.43569685 -0.81500814  0.20745268 -0.69211872 -0.63175355
 -0.29409296 -0.63062904]
number of zero coefficients: 0

lambda: 1
coefficients:
[52.7675      8.07360599  4.85154708 -3.06404228 -4.51372277  1.27840121
 -0.65920214 -0.65018018 -0.58940495  0.33535769 -0.62922629 -0.71269362
 -0.25309823 -0.36610874]
number of zero coefficients: 0

lambda: 10
coefficients:
[52.7675      5.57342379  3.54860721 -1.98802717 -2.96120009  0.19238397
  0.11802234 -1.30889107  0.19452634  0.8976363  -0.42652759 -0.94834877
 -0.17311282  0.56710226]
number of zero coefficients: 0

lambda: 100
coefficients:
[ 5.27675000e+01  1.88902560e+00  1.06651781e+00 -5.54281721e-01
 -8.36212761e-01 -2.73180240e-01  6.90061269e-03 -8.81867413e-01
  3.77486718e-01  8.23151396e-01 -1.64442623e-01 -4.72144148e-01
 -6.00127859e-02  6.07605533e-01]
number of zero coefficients: 0



In [10]:
theta_ridge_0 = fit_ridge(
    X_train_std,
    y_train,
    0
)

theta_ols_std = fit_normal_equation(
    X_train_std,
    y_train
)

print(np.allclose(theta_ridge_0, theta_ols_std))


True


## Task 3

In [12]:
def k_fold_cv(x,y,lam,k=5,seed=0):

    n = len(y)

    idx = np.random.default_rng(seed).permutation(n)

    folds = np.array_split(idx,k)
    fold_rmse = []

    for i,val_idx in enumerate(folds):

        train_idx = np.concatenate(folds[:i]+folds[i+1:])

        theta = fit_ridge(x[train_idx],y[train_idx],lam)

        y_pred = design_matrix(x[val_idx])@theta
        fold_rmse.append(rmse(y[val_idx],y_pred))

    return np.mean(fold_rmse)


In [13]:
lams_cv = [0, 0.1, 0.3, 1, 3, 10, 30, 100]
cv_rmse = []

print(f'{"lambda":>10}{"CV RMSE":>15}')

for lam in lams_cv:
    score = k_fold_cv(
        X_train_std,
        y_train,
        lam,
        k=5,
        seed=0
    )

    cv_rmse.append(score)
    print(f"{lam:10.1f}{score:15.4f}")

best_idx = np.argmin(cv_rmse)
best_lambda = lams_cv[best_idx]
best_cv_rmse = cv_rmse[best_idx]

print()
print("best lambda:", best_lambda)
print("best CV RMSE:", best_cv_rmse)


    lambda        CV RMSE
       0.0         3.2686
       0.1         3.2006
       0.3         3.0929
       1.0         2.9396
       3.0         3.2751
      10.0         4.7113
      30.0         6.4775
     100.0         8.2539

best lambda: 1
best CV RMSE: 2.939599585571018


In [14]:
theta_best = fit_ridge(
    X_train_std,
    y_train,
    best_lambda
)

A_test = design_matrix(X_test_std)
y_test_pred = A_test @ theta_best

test_rmse_best = rmse(
    y_test,
    y_test_pred
)

theta_0 = fit_ridge(
    X_train_std,
    y_train,
    0
)

y_test_pred_0 = A_test @ theta_0

test_rmse_0 = rmse(
    y_test,
    y_test_pred_0
)

print("test RMSE at lambda = 0:", test_rmse_0)
print("test RMSE at chosen lambda:", test_rmse_best)


test RMSE at lambda = 0: 3.342996557689886
test RMSE at chosen lambda: 3.0856646635989136


## Homework H1

In [ ]:
theta_std = fit_ridge(
    X_train_std,
    y_train,
    0
)

w_raw = theta_std[1:] / sigma
b_raw = theta_std[0] - np.sum(
    theta_std[1:] * mu / sigma
)

theta_raw_from_std = np.r_[b_raw, w_raw]

print("coefficients match:", np.allclose(
    theta_raw_from_std,
    theta_raw
))

pred_raw = design_matrix(X_train) @ theta_raw
pred_from_std = design_matrix(X_train_std) @ theta_std

print("predictions match:", np.allclose(
    pred_raw,
    pred_from_std
))


## Homework H2

In [ ]:
path_lams = np.logspace(-2, 3, 100)
path = []

for lam in path_lams:
    path.append(
        fit_ridge(
            X_train_std,
            y_train,
            lam
        )
    )

path = np.array(path)

plt.figure(figsize=(9, 6))

for j in range(5):
    plt.plot(
        path_lams,
        path[:, j + 1],
        label=features[j]
    )

for j in range(5, len(features)):
    plt.plot(
        path_lams,
        path[:, j + 1],
        color="gray",
        alpha=0.7,
        label=features[j]
    )

plt.xscale("log")
plt.xlabel("lambda")
plt.ylabel("coefficient")
plt.title("Ridge Coefficient Path")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

print("minimum absolute coefficient for each feature:")
for j, name in enumerate(features):
    print(name, np.min(np.abs(path[:, j + 1])))


## Homework H3

In [ ]:
positive_lams = np.array(lams_cv[1:], dtype=float)
positive_cv = np.array(cv_rmse[1:])

plt.figure(figsize=(8, 5))
plt.plot(positive_lams, positive_cv, marker="o")
plt.scatter(
    best_lambda,
    best_cv_rmse,
    marker="o"
)

plt.xscale("log")
plt.xlabel("lambda")
plt.ylabel("CV RMSE")
plt.title("Cross-validation Curve")
plt.show()

print("best lambda:", best_lambda)


## Homework H4

In [ ]:
def soft_threshold(v, t):
    return np.sign(v) * np.maximum(np.abs(v) - t, 0)


def fit_lasso(X, y, lam, lr=0.05, n_iters=30000):
    A = design_matrix(X)
    n = len(y)

    theta = np.zeros(A.shape[1])

    for _ in range(n_iters):
        error = A @ theta - y
        gradient = (2 / n) * (A.T @ error)

        theta = theta - lr * gradient
        theta[1:] = soft_threshold(
            theta[1:],
            lr * lam
        )

    return theta


## Homework H5

In [ ]:
lasso_lams = [0.1, 0.5, 1.0, 2.0]

lasso_results = []

for lam in lasso_lams:
    theta_lasso = fit_lasso(
        X_train_std,
        y_train,
        lam,
        lr=0.05,
        n_iters=30000
    )

    zero_mask = np.abs(theta_lasso[1:]) < 1e-8
    zero_count = np.sum(zero_mask)

    eliminated = [
        features[i]
        for i in range(len(features))
        if zero_mask[i]
    ]

    y_pred = design_matrix(X_test_std) @ theta_lasso
    test_score = rmse(y_test, y_pred)

    lasso_results.append(
        (lam, theta_lasso, test_score, eliminated)
    )

    print("lambda:", lam)
    print("zero coefficients:", zero_count)
    print("eliminated:", eliminated)
    print("test RMSE:", test_score)
    print()


In [ ]:
ridge_theta = fit_ridge(
    X_train_std,
    y_train,
    best_lambda
)

print(f'{"feature":14s}{"ridge":>14}{"lasso":>14}')

for i, name in enumerate(features):
    lasso_theta = lasso_results[-1][1]
    print(
        f"{name:14s}"
        f"{ridge_theta[i + 1]:14.6f}"
        f"{lasso_theta[i + 1]:14.6f}"
    )

ridge_test_rmse = rmse(
    y_test,
    design_matrix(X_test_std) @ ridge_theta
)

lasso_test_rmse = lasso_results[-1][2]

print()
print("ridge lambda:", best_lambda)
print("ridge test RMSE:", ridge_test_rmse)
print("lasso lambda:", lasso_results[-1][0])
print("lasso test RMSE:", lasso_test_rmse)
